In [ ]:
import re
import os
import pandas as pd

def convert_vissim_rsr_to_excel(rsr_file_path, excel_file_path):
    """
    Convert .rsr file to xlsx file. This function reads a Vissim .rsr result file,
    extracts relevant data lines, and saves them into an Excel file. It also attempts to convert columns to numeric types
    to align with the pandas latest style.
    
    :param rsr_file_path: input .rsr file path
    :param excel_file_path: output .xlsx file path
    :return: True if conversion is successful, False otherwise
    """
    try:
        data_lines = []
        # Vissim result files are usually encoded in cp949.
        with open(rsr_file_path, 'r', encoding='cp949', errors='ignore') as f:
            for line in f:
                # Starting numbered lines with semicolon are data lines.
                if re.match(r'^\s*\d+\.\d+;', line.strip()):
                    data_lines.append(line.strip())

        if not data_lines:
            print(f"Not Found valid data lines in '{rsr_file_path}' file.")
            return False

        header = ['Time', 'No', 'Veh', 'VehType', 'Trav', 'Delay', 'Dist']

        parsed_data = []
        for line in data_lines:
            values = [v.strip() for v in line.split(';')]
            if values and values[-1] == '':
                values.pop()

            try:
                row_dict = {header[i]: values[i] for i in range(len(values))}
                parsed_data.append(row_dict)
            except IndexError:
                print(f"IndexError: Incorrect number of values in line: {line}")

        df = pd.DataFrame(parsed_data)

        # Attempt to convert columns to numeric types where possible
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except ValueError:
                # If conversion fails, keep the column as is (likely string)
                pass

        df.to_excel(excel_file_path, index=False)
        print(f"✅ Successfully converted '{rsr_file_path}' to Excel format.")
        return True

    except FileNotFoundError:
        print(f"❌ Error: '{rsr_file_path}' file not found.")
        return False
    except Exception as e:
        print(f"❌ Error: An error occurred during file conversion: {e}")
        return False

def analyze_vissim_output(excel_file_path, log_file_path='log.txt'):
    """
    Analyze the converted Excel file to compute average speeds and traffic volumes,
    and save the results to a log.txt file.

    :param excel_file_path: Path to the Excel file to be analyzed
    :param log_file_path: Path to the log file where results will be saved
    """
    try:
        df = pd.read_excel(excel_file_path)
        print(f"✅ Loaded Excel file: '{excel_file_path}'")

        # Initialize results dictionary
        results = {}

        # 1. Average Speed Calculation (kph)
        # Speed = (Distance(m) / Travel Time(s)) * 3.6
        speed_conditions = {
            'a0_speed': [1, 2, 3],
            'a1_speed': [4, 5, 6, 7, 8],
            'a2_speed': [9, 10, 11, 12, 13],
            'a3_speed': [14, 15, 16, 17, 18],
            'a4_speed': [19, 20, 21],
            'a5_speed': [22, 23, 24, 25, 26]
        }

        for name, no_list in speed_conditions.items():
            # Filter rows corresponding to the specified 'No' values
            subset = df[df['No'].isin(no_list)].copy()
            # Calculate speed only for rows where travel time (Trav) is greater than 0
            subset = subset[subset['Trav'] > 0]

            if not subset.empty:
                subset['Speed_kmh'] = (subset['Dist'] / subset['Trav']) * 3.6
                avg_speed = subset['Speed_kmh'].mean()
                results[name] = avg_speed
            else:
                results[name] = 0.0 # If no valid data, set average speed to 0

        # 2. Traffic Volume Calculation (vph)
        # Count the number of vehicles for each 'No' condition and convert to vehicles per hour (vph)
        volume_conditions = {
            'Q0_vph': [1],
            'Q1_vph': [2, 3],
            'Q2_vph': [7],
            'Q3_vph': [5],
            'Q4_vph': [4, 6, 8],
            'Q5_vph': [12, 13],
            'Q6_vph': [9, 11],
            'Q7_vph': [10],
            'Q8_vph': [17],
            'Q9_vph': [14, 16],
            'Q10_vph': [15, 18],
            'Q11_vph': [19],
            'Q12_vph': [20, 21],
            'Q13_vph': [24, 26],
            'Q14_vph': [22, 25],
            'Q15_vph': [23]
        }

        for name, no_list in volume_conditions.items():
            count = len(df[df['No'].isin(no_list)])
            results[name] = count

        # 3. Sum of Input Traffic Volumes
        results['input0'] = results.get('Q0_vph', 0) + results.get('Q1_vph', 0)
        results['input1'] = results.get('Q2_vph', 0) + results.get('Q3_vph', 0) + results.get('Q4_vph', 0)
        results['input2'] = results.get('Q5_vph', 0) + results.get('Q6_vph', 0) + results.get('Q7_vph', 0)
        results['input3'] = results.get('Q8_vph', 0) + results.get('Q9_vph', 0) + results.get('Q10_vph', 0)
        results['input4'] = results.get('Q11_vph', 0) + results.get('Q12_vph', 0)
        results['input5'] = results.get('Q13_vph', 0) + results.get('Q14_vph', 0) + results.get('Q15_vph', 0)

        # Ensure the log file directory exists
        with open(log_file_path, 'w', encoding='utf-8') as f:
            f.write(f"filename:{log_file_path} ===\n")
            f.write("--- vissim_1_no Average Speed Analysis (km/h) ---\n")
            for name, _ in speed_conditions.items():
                f.write(f"average {name}: {results.get(name, 0):.2f}\n")

            f.write("\n--- Traffic Volume Analysis (vph) ---\n")
            # Print traffic volumes Q0 to Q15 in order
            for i in range(16):
                name = f"Q{i}_vph"
                log_name = name
                f.write(f"{log_name}: {results.get(name, 0)}\n")

            f.write("\n--- Output Volume Summary (vph) ---\n")
            for i in range(6):
                name = f"input{i}"
                f.write(f"Output{i}: {results.get(name, 0)}\n")

        print(f"✅ Analysis complete! Results saved to '{log_file_path}'.")

    except FileNotFoundError:
        print(f"❌ Error: The Excel file '{excel_file_path}' to be analyzed was not found.")
    except KeyError as e:
        print(f"❌ Error: The Excel file is missing required columns ({e}). Please check the file format.")
    except Exception as e:
        print(f"❌ Error: An error occurred during analysis: {e}")



## Set the .rsr input folder path

In [ ]:
test_dir = 'YOUR_RSR_FILE_DIRECTORY'

print(f"rsr 파일 : {len(os.listdir(test_dir))}")

In [ ]:
if __name__ == "__main__":
  # 0. Set the .rsr input folder path
  # example: test_dir = 'YOUR_RSR_FILE_DIRECTORY'

  test_dir = test_dir
  print(f"rsr folder : {test_dir}")
  print(f"Count rsr files: {len(os.listdir(test_dir))}")

  num = 0
  for file in os.listdir(test_dir):
    input_rsr_file = os.path.join(test_dir, file)
    print(f"{num} \n✅ Input rsr file! {input_rsr_file}")

    # 1. Generate the output Excel file path automatically (.rsr -> .xlsx)
    output_excel_file = input_rsr_file.replace('.rsr', '.xlsx')

    # 2. Convert the RSR file to an Excel file.
    conversion_successful = convert_vissim_rsr_to_excel(input_rsr_file, output_excel_file)

    # 3. Run the analysis only if the conversion was successful.
    if conversion_successful:
      # 4. Save the results to log.txt from the converted Excel file.
      analyze_vissim_output(output_excel_file, log_file_path=output_excel_file.replace('.xlsx', '.txt'))
    else:
      print(f"🔥 Analysis failed! {output_excel_file}")
    num+=1

## Mean and standard error of all individual items (16+6+6+6) for random seeds

In [ ]:
import os
import re
import numpy as np
from collections import defaultdict

# configuration
# 1. Path to the folder where data files are stored
TARGET_DIRECTORY = test_dir

# traget_directory/result
if not os.path.exists(f'{TARGET_DIRECTORY}/result'):
  os.makedirs(f'{TARGET_DIRECTORY}/result')
  print(f"✅ Created result folder '{f'{TARGET_DIRECTORY}/result'}'.")

# 2. Name of the file to save analysis results
OUTPUT_FILENAME = f'{TARGET_DIRECTORY}/result/analysis_summary.txt'

# 3. List of keywords to analyze
KEYWORDS = [
    'average a0_speed', 'average a1_speed', 'average a2_speed',
    'average a3_speed', 'average a4_speed', 'average a5_speed',
    'Q0_vph', 'Q1_vph', 'Q2_vph', 'Q3_vph', 'Q4_vph', 'Q5_vph',
    'Q6_vph', 'Q7_vph', 'Q8_vph', 'Q9_vph', 'Q10_vph', 'Q11_vph',
    'Q12_vph', 'Q13_vph', 'Q14_vph', 'Q15_vph'
]


def natural_sort_key(s):
    """
    natural sort key function
    Ordering function for "natural sort". (e.g, "item1", "item2", "item10")
    """
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

def analyze_text_files(directory, keywords, output_file):
    """
    Analyze all .txt files in the specified directory to compute statistics for each keyword,
    and output the results to both the console and a text file.
    """
    data_collector = defaultdict(list)
    processed_files_count = 0

    try:
        file_list = [f for f in os.listdir(directory) if f.endswith('.txt')]
        if not file_list:
            print(f"Error: Not Found .txt files to analyze in '{directory}' folder.")
            return

        print(f"Analyzing {len(file_list)} files...\n")

        for filename in file_list:
            file_path = os.path.join(directory, filename)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        parts = line.strip().split(':')
                        if len(parts) == 2:
                            key = parts[0].strip()
                            value_str = parts[1].strip()
                            if key in keywords:
                                try:
                                    value = float(value_str)
                                    data_collector[key].append(value)
                                except ValueError:
                                    print(f"Warning: Could not convert value '{value_str}' for key '{key}' in file '{filename}' to float.")
                processed_files_count += 1
            except Exception as e:
                print(f"Error: An issue occurred while reading file '{filename}': {e}")

        # --- Save and output results ---
        try:
            with open(output_file, 'w', encoding='utf-8') as outfile:
                header = f"--- final analysis results (Total {processed_files_count} files) ---\n"
                print(f"\n{header.strip()}")
                outfile.write(header + "\n")

                # Order keys naturally
                sorted_keys = sorted(data_collector.keys(), key=natural_sort_key)

                for key in sorted_keys:
                    values = data_collector[key]
                    output_line = ""
                    if len(values) > 1:
                        values_np = np.array(values)
                        mean = np.mean(values_np)
                        std_dev = np.std(values_np, ddof=1)
                        count = len(values)
                        output_line = f"item: {key:<20} | file count: {count:<4} | mean: {mean:>8.2f} | std dev (n-1): {std_dev:>8.2f}\n"
                    elif values:
                        mean = values[0]
                        count = len(values)
                        output_line = f"item: {key:<20} | file count: {count:<4} | mean: {mean:>8.2f} | std dev (n-1): {'N/A':>8}\n"
                    else:
                        output_line = f"item: {key:<20} | No data found.\n"

                    # Log to both console and file
                    print(output_line.strip())
                    outfile.write(output_line)

            print(f"✅ Analysis complete! Results saved to '{output_file}'.")

        except Exception as e:
            print(f"Error: An issue occurred while saving results to file '{output_file}': {e}")

    except FileNotFoundError:
        print(f"Error: Not Found .txt files to analyze in '{directory}' folder.")
    except Exception as e:
        print(f"Error: An unexpected error occurred during analysis: {e}")


if __name__ == "__main__":
    analyze_text_files(TARGET_DIRECTORY, KEYWORDS, OUTPUT_FILENAME)

## Input None-peak Ground Truth data

In [ ]:
# Input non-peak Ground Truth data
# 16 traffic volume (vph)
ACTUAL_VOLUMES = [59, 104, 130, 71, 36, 42, 124, 10, 86, 33, 6, 103, 23, 79, 77, 29]
# 6 per-approach speeds (kph)
ACTUAL_SPEEDS = [31, 36, 26, 39, 32, 20]

## MSE based on the average of all random seeds

In [ ]:
import re
import numpy as np
from sklearn.metrics import mean_squared_error
import math

INPUT_FILENAME = f'{TARGET_DIRECTORY}/result/analysis_summary.txt'
OUTPUT_FILENAME = f'{TARGET_DIRECTORY}/result/mse_results.txt'

def parse_summary_file(filename):
    """
    Return a dictionary of average values for each item by parsing the analysis summary file.
    """
    data = {}
    # Find the pattern 'item: (key) | ... mean: (value)' using regular expressions.
    pattern = re.compile(r"item:\s*(.*?)\s*\|.*?mean:\s*([0-9.]+)")

    try:
        with open(filename, 'r', encoding='utf-8') as f:
            for line in f:
                match = pattern.search(line)
                if match:
                    key = match.group(1).strip()
                    value = float(match.group(2))
                    data[key] = value
        print(f"✅ '{filename}' file parsed. Found a total of {len(data)} items.")
        return data
    except FileNotFoundError:
        print(f"Error: Not Found '{filename}' file.")
        return None
    except Exception as e:
        print(f"Error: An error occurred while reading the file: {e}")
        return None

def calculate_and_print_mse(title, predicted, actual, file_handle=None):
    """
    Calculate and print the Mean Squared Error (MSE) between predicted and actual values.
    """
    if len(predicted) != len(actual):
        error_msg = f"Error: The number of predicted values ({len(predicted)}) and actual values ({len(actual)}) for '{title}' do not match."
        print(error_msg)
        if file_handle:
            file_handle.write(error_msg + "\n")
        return

    mse = mean_squared_error(actual, predicted)

    # Prepare output lines
    output_lines = [
        f"--- {title} ---",
        f"  - predicted value: {[math.trunc(p) for p in predicted]}",
        f"  - actual value: {actual}",
        f"  - MSE: {mse:.4f}\n"
    ]

    # Log to both console and file
    for line in output_lines:
        print(line)
        if file_handle:
            file_handle.write(line + "\n")


if __name__ == "__main__":
    # Parse the predicted data from the summary file
    predicted_data = parse_summary_file(INPUT_FILENAME)

    if predicted_data:
        # Open the file to save results
        try:
            with open(OUTPUT_FILENAME, 'w', encoding='utf-8') as outfile:
                header = [
                    "="*50,
                    "MSE (Mean Squared Error) Calculation Results",
                    "="*50,
                    ""
                ]
                for line in header:
                    print(line)
                    outfile.write(line + "\n")
                
                # 3. Traffic Volume MSE calculation
                try:
                    predicted_volumes = [predicted_data[f'Q{i}_vph'] for i in range(16)]
                    calculate_and_print_mse("Traffic Volume MSE", predicted_volumes, ACTUAL_VOLUMES, file_handle=outfile)
                except KeyError as e:
                    print(f"Error: Key error while retrieving traffic volume data - {e}. Please ensure all Q_vph entries are present in the file.")

                # 4. Speed MSE calculation
                try:
                    predicted_speeds = [predicted_data[f'average a{i}_speed'] for i in range(6)]
                    calculate_and_print_mse("Speed MSE", predicted_speeds, ACTUAL_SPEEDS, file_handle=outfile)

                except KeyError as e:
                    print(f"Error: Key error while retrieving speed data - {e}. Please ensure all average a_speed entries are present in the file.")

            print(f"\nSuccessfully saved analysis results to '{OUTPUT_FILENAME}' file.")

        except Exception as e:
            print(f"Error: An issue occurred while saving results to file '{OUTPUT_FILENAME}': {e}")

## Individual random seed MSE analysis

In [ ]:
import os
import re
import numpy as np
from sklearn.metrics import mean_squared_error
import math

OUTPUT_FILENAME = f'{TARGET_DIRECTORY}/result/individual_mse_with_summary.txt'

def parse_simulation_file(filepath):
    """
    Parse a single simulation result file and return a dictionary of key-value pairs.
    """
    data = {}
    # Find the pattern: 'aveage a0_speed: 33.59' or 'Q0_vph: 128'
    pattern = re.compile(r"^(.*?):\s*([0-9.]+)")

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                match = pattern.search(line.strip())
                if match:
                    key = match.group(1).strip()
                    value = float(match.group(2))
                    data[key] = value
        return data
    except Exception as e:
        print(f"Error: An issue occurred while parsing file '{filepath}': {e}")
        return None

if __name__ == "__main__":
    # Initialize lists to store each MSE result
    volume_mses = []
    speed1_mses = []

    # Initialize save buffer for detailed results of each file
    detailed_results_buffer = []

    try:
        # Create the output directory if it doesn't exist
        os.makedirs(f'{TARGET_DIRECTORY}/result', exist_ok=True)

        # 1. Read and sort the list of .txt files in the target directory
        file_list = sorted([f for f in os.listdir(TARGET_DIRECTORY) if f.endswith('.txt')])
        if not file_list:
            print(f"Error: Not Found .txt files to analyze in '{TARGET_DIRECTORY}' folder.")
            exit()

        print(f"Analyzing {len(file_list)} files...\n")

        # 2. Start analyzing each file and temporarily store results in memory.
        for filename in file_list:
            filepath = os.path.join(TARGET_DIRECTORY, filename)

            # Skip the output file itself to prevent
            if os.path.abspath(filepath) == os.path.abspath(OUTPUT_FILENAME):
                continue

            print(f"-> analysis in progress: {filename}")

            predicted_data = parse_simulation_file(filepath)

            # Initialize String list to store detailed results of the current file
            file_detail_string = [f"\n\n{'='*60}\nanalysis file: {filename}\n{'='*60}"]

            if not predicted_data:
                file_detail_string.append("\n\nError: Failed to parse the file.\n")
                detailed_results_buffer.append("".join(file_detail_string))
                continue

            # 3. Traffic Volume MSE calculation and storage    
            try:
                predicted_volumes = [predicted_data[f'Q{i}_vph'] for i in range(16)]
                mse_vol = mean_squared_error(ACTUAL_VOLUMES, predicted_volumes)
                volume_mses.append(mse_vol)
                file_detail_string.append(
                    f"\n--- Traffic Volume (Volume) MSE ---\n"
                    f"  - MSE: {mse_vol:.4f}\n"
                    f"  - predicted value: {[math.trunc(p) for p in predicted_volumes]}\n"
                    f"  - actual value: {ACTUAL_VOLUMES}"
                )
            except (KeyError, IndexError) as e:
                volume_mses.append(float('nan')) # Add NaN if data is missing
                error_msg = f"\nError: An issue occurred while retrieving traffic volume data - {e}."
                file_detail_string.append(error_msg)
                print(f"Error: {filename} data error: {e}")

            # 4. Speed MSE calculation and storage
            try:
                predicted_speeds = [predicted_data[f'average a{i}_speed'] for i in range(6)]

                mse_s1 = mean_squared_error(ACTUAL_SPEEDS, predicted_speeds)

                speed1_mses.append(mse_s1)

                file_detail_string.append(
                    f"\n\n--- Speed MSE ---\n"
                    f"  - MSE: {mse_s1:.4f}\n"
                    f"  - predicted value: {[math.trunc(p) for p in predicted_speeds]}\n"
                    f"  - actual value: {ACTUAL_SPEEDS}"
                )

            except (KeyError, IndexError) as e:
                speed1_mses.append(float('nan'))
                error_msg = f"\nError: An issue occurred while retrieving speed data - {e}."
                file_detail_string.append(error_msg)
                print(f"Error: {filename} speed data error: {e}")

            detailed_results_buffer.append("".join(file_detail_string))

        # 5. After all analyses are complete, write the results to the file at once.
        with open(OUTPUT_FILENAME, 'w', encoding='utf-8') as outfile:
            # 5.1 Write summary information
            summary_header = [
                "="*60,
                " Summary of MSE for",
                "="*60,
                ""
            ]
            outfile.write("\n".join(summary_header))

            volume_mses_rounded = [round(mse, 4) for mse in volume_mses]
            speed1_mses_rounded = [round(mse, 4) for mse in speed1_mses]

            outfile.write(f"Traffic Volume MSE: {volume_mses_rounded}\n")
            outfile.write(f"Speed MSE: {speed1_mses_rounded}\n")

            # 5.2 Write detailed results for each file
            detailed_header = [
                "\n\n",
                "="*60,
                " Individual File Detailed Results",
                "="*60,
            ]
            outfile.write("\n".join(detailed_header))
            outfile.write("".join(detailed_results_buffer))

        print(f"\n✅ Successfully saved analysis results to '{OUTPUT_FILENAME}' file.")

    except Exception as e:
        import traceback
        print(f"Error: An unexpected error occurred during analysis. {e}")
        traceback.print_exc()